# 第 7 章 · 多 agent（CrewAI · Expert Teams）

两个角色（研究员 + 撰稿人）串成一条内容生产流水线。

关键在 `Task(context=[research_task])`：**后一个 task 能把前一个 task 的输出当上下文**。
这就是 CrewAI 版的"顺序交接"。

kernel 选 `Python (agentic-crewai)`。

In [1]:
import os
import threading

from crewai import Agent, Task, Crew, Process, LLM

# 【适配】书里用 ChatGoogleGenerativeAI("gemini-2.0-flash")；
#   CrewAI 1.15 不收 LangChain 对象，改用内置 provider，指向 DeepSeek
llm = LLM(model="deepseek/deepseek-flash",
          api_key=os.environ["OPENAI_API_KEY"],
          base_url="https://api.deepseek.com/v1")

researcher = Agent(
    role='Senior Research Analyst',
    goal='Find and summarize the latest trends in AI.',
    backstory="You are an experienced research analyst with a knack for identifying key trends.",
    verbose=False,
    allow_delegation=False,
    llm=llm,
)

writer = Agent(
    role='Technical Content Writer',
    goal='Write a clear and engaging blog post based on research findings.',
    backstory="You are a skilled writer who can translate complex technical topics into accessible content.",
    verbose=False,
    allow_delegation=False,
    llm=llm,
)

research_task = Task(
    description="Research the top 3 emerging trends in AI. Focus on practical applications.",
    expected_output="A detailed summary of the top 3 AI trends.",
    agent=researcher,
)

# ★ 本章新增 context：把上一个 task 的结果喂给这个 task —— CrewAI 的"交接"
writing_task = Task(
    description="Write a short blog post (about 200 words) based on the research findings.",
    expected_output="A complete blog post about the latest AI trends.",
    agent=writer,
    context=[research_task],
)

blog_creation_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
)


# 【适配】kernel 里有运行中的事件循环，同步 kickoff() 会抛 RuntimeError → 独立线程（同第 5 章）
def run_in_thread(fn):
    box = {}

    def _target():
        try:
            box["result"] = fn()
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


print("## Running the blog creation crew ##")
result = run_in_thread(blog_creation_crew.kickoff)
print("\n## Final Output ##\n")
print(str(result)[:1200])

## Running the blog creation crew ##



## Final Output ##

# Beyond Chatbots: Three AI Trends Moving Into Production

AI is shifting from clever demos to operational systems. The most practical near-term trends are agentic AI, multimodal vertical copilots, and small or edge models.

**1. Agentic AI: from answers to action.** These systems plan, retrieve information, call tools or APIs, and execute multi-step workflows. Most successful deployments are “bounded agents” with clear permissions, audit logs, and human approval for consequential actions. Practical uses include support ticket triage, IT alert analysis, invoice matching, code review, and CRM enrichment.

**2. Multimodal and vertical copilots.** AI can now work across text, images, audio, video, and documents—embedded directly in industry workflows. High-ROI examples include clinical documentation, insurance claims extraction, visual manufacturing inspection, field-service diagnosis, and contract review. Human review remains essential in regulated, high-stakes setti